# ImmigrationNavigator — MVP App

**UC Berkeley MIDS Capstone 2026** — Team: Ale, Clover, Duc, Rohan

---

## What this notebook does

Launches the conversational UI using the merged RAG pipeline.
This is the user-facing product — everything from Notebooks 1–3 comes together here.

## Features

- **Conversational chat** — ask questions in natural language
- **Situation profiling** — visa status, degree field, graduation date, employer type
- **Synonym expansion** — user terms automatically mapped to legal equivalents
- **Topic reranking** — F-1/OPT/H-1B results boosted, irrelevant visa categories penalized
- **Deadline calculator** — personalized OPT/STEM OPT/H-1B dates appended automatically
- **Citation enforcement** — every claim cites its USCIS source
- **Legal disclaimer** — clear notice that this is guidance, not legal advice

## How to run

Run all cells top to bottom. The last cell prints a public URL you can share for user testing.

## Prerequisites

- Notebook 3 must have been run at least once (builds the ChromaDB vector store)
- Set `USE_OPENAI` below to match the model you chose after benchmarking
- `synonyms.py` must be in `/home/sagemaker-user/`

## 1. Install Dependencies

In [1]:
!pip install langchain langchain-groq langchain-community langchain-openai \
             chromadb fastembed gradio boto3 -q

## 2. Imports and Model Configuration

Set `USE_OPENAI` to match whichever model performed better in the Notebook 3 benchmark.

All credentials are loaded from AWS Secrets Manager — no API keys in code.

In [2]:
import os, re, json, uuid, boto3
from datetime import datetime, date, timedelta
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from fastembed import TextEmbedding
import chromadb
import gradio as gr

USE_OPENAI = False  # ← set to True once you've decided from benchmark

def get_secret(name):
    client = boto3.client("secretsmanager", region_name="us-east-1")
    return json.loads(client.get_secret_value(SecretId=name)["SecretString"])

secrets = get_secret("immigration-navigator/groq")
os.environ["GROQ_API_KEY"] = secrets["GROQ_API_KEY"]

if USE_OPENAI:
    openai_s = get_secret("immigration-navigator/openai")
    os.environ["OPENAI_API_KEY"] = openai_s["OPENAI_API_KEY"]

s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"
print("✅ Setup complete")

2026-06-12 22:35:59.458292729 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


✅ Setup complete


## 3. Load Models and Vector Store

Connects to the ChromaDB collection built in Notebook 3.
No re-embedding needed — the vector store persists across sessions.

Also loads `synonyms.py` if available — the app works without it but query expansion significantly improves retrieval quality.

In [3]:
collection_name = f"immigration_nav_{'openai' if USE_OPENAI else 'groq'}"

if USE_OPENAI:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    embed_fn = OpenAIEmbeddings(model="text-embedding-3-small")
    def get_embedding(texts): return embed_fn.embed_documents(texts)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print("✅ Using OpenAI")
else:
    embed_model = TextEmbedding("BAAI/bge-small-en-v1.5")
    def get_embedding(texts): return [e.tolist() for e in embed_model.embed(texts)]
    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=secrets["GROQ_API_KEY"], temperature=0)
    print("✅ Using Groq")

chroma_client = chromadb.PersistentClient(path="/home/sagemaker-user/chroma_db")
collection = chroma_client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
print(f"✅ ChromaDB: {collection.count()} chunks")

try:
    import sys; sys.path.append('/home/sagemaker-user')
    from synonyms import expand_query
    HAS_SYNONYMS = True
    print("✅ synonyms.py loaded")
except ImportError:
    HAS_SYNONYMS = False
    print("⚠️  synonyms.py not found")

✅ Using Groq
✅ ChromaDB: 3066 chunks
✅ synonyms.py loaded


## 4. Deadline Calculator

Deterministic rules-based module — completely separate from the LLM.

All rules sourced from USCIS Policy Manual, Volume 2 Part F Chapter 5:
- OPT application window: 90 days before to 60 days after graduation
- OPT period: 12 months, max 90 days unemployment
- STEM OPT: apply 90 days before OPT expires, 24-month extension
- H-1B: approximate annual lottery and start dates

> **Why separate from the LLM?** Date arithmetic is too important to leave to a language model. The LLM narrates; the rules engine computes.

In [4]:
def calculate_deadlines(graduation_date: date) -> dict:
    """Rules-based deadline calculator. No LLM. USCIS Policy Manual Vol. 2 Part F Ch. 5."""
    return {
        "opt_application": {
            "earliest": graduation_date - timedelta(days=90),
            "latest":   graduation_date + timedelta(days=60),
            "note":     "File Form I-765 within this window"
        },
        "opt_period": {
            "start": graduation_date,
            "end":   graduation_date + timedelta(days=365),
            "note":  "12 months of OPT. Max 90 days unemployment."
        },
        "stem_opt_application": {
            "deadline": graduation_date + timedelta(days=365) - timedelta(days=90),
            "note":     "File Form I-765 + I-983. Apply at least 90 days before OPT expires."
        },
        "stem_opt_period": {
            "start": graduation_date + timedelta(days=365),
            "end":   graduation_date + timedelta(days=365+730),
            "note":  "24-month STEM OPT extension. Max 150 days unemployment."
        },
        "h1b_timeline": {
            "lottery_opens":  date(graduation_date.year, 3, 1),
            "petition_filed": date(graduation_date.year, 4, 1),
            "start_date":     date(graduation_date.year, 10, 1),
            "note":           "Cap-gap extends F-1 until Oct 1 if H-1B pending."
        }
    }

def format_deadlines(d: dict) -> str:
    lines = [
        "📅 **Your Immigration Timeline**", "",
        "**OPT Application Window**",
        f"- Earliest: {d['opt_application']['earliest'].strftime('%B %d, %Y')}",
        f"- Latest:   {d['opt_application']['latest'].strftime('%B %d, %Y')}",
        f"- {d['opt_application']['note']}", "",
        "**OPT Period**",
        f"- Start: {d['opt_period']['start'].strftime('%B %d, %Y')}",
        f"- End:   {d['opt_period']['end'].strftime('%B %d, %Y')}",
        f"- {d['opt_period']['note']}", "",
        "**STEM OPT Extension**",
        f"- Apply by: {d['stem_opt_application']['deadline'].strftime('%B %d, %Y')}",
        f"- {d['stem_opt_application']['note']}", "",
        "**H-1B Timeline**",
        f"- Lottery opens:  {d['h1b_timeline']['lottery_opens'].strftime('%B %d, %Y')}",
        f"- Petition filed: {d['h1b_timeline']['petition_filed'].strftime('%B %d, %Y')}",
        f"- H-1B starts:   {d['h1b_timeline']['start_date'].strftime('%B %d, %Y')}",
        f"- {d['h1b_timeline']['note']}",
    ]
    return "\n".join(lines)

print("✅ Deadline calculator ready")

✅ Deadline calculator ready


## 5. RAG Pipeline

The `ask()` function runs the full pipeline:

1. **Synonym expansion** — normalize user query to legal terms
2. **Profile injection** — add visa status, degree, employer type to query
3. **Vector retrieval** — top-5 most similar chunks from ChromaDB
4. **Topic reranking** — boost immigration-relevant chunks, penalize off-topic ones
5. **LLM generation** — cited, personalized answer

The deadline calculator is called separately in the Gradio `chat()` function when the question contains date-related keywords.

In [5]:
PROMPT = ChatPromptTemplate.from_template("""
You are ImmigrationNavigator, an AI assistant helping international students
navigate U.S. visa processes. Answer ONLY using the provided context.
Cite every claim with [Source: label, url].
If context is insufficient, say:
"I don't have enough information. Please consult your ISO or an immigration attorney."

User Profile:
- Visa status: {visa_status}
- Degree field: {degree_field}
- Graduation date: {graduation_date}
- Employer type: {employer_type}

Context:
{context}

Question: {question}

Answer (personalized, cite every claim):
""")

RELEVANT   = ["f-1", "opt", "stem opt", "h-1b", "cap-gap", "ead", "i-765", "sevis"]
IRRELEVANT = ["j-1", "b-1", "h-2b", "l-1", "o-1", "adjustment of status", "naturalization"]

def ask(question: str, profile: dict, n_results: int = 5) -> str:
    expanded = expand_query(question) if HAS_SYNONYMS else question
    enriched = f"{expanded} | visa: {profile.get('visa_status','')} | degree: {profile.get('degree_field','')} | employer: {profile.get('employer_type','')}"

    embeddings = get_embedding([enriched])
    results = collection.query(query_embeddings=embeddings, n_results=n_results, include=["documents", "metadatas", "distances"])

    reranked = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        score = 1 - dist
        text_lower = doc.lower()
        rel = sum(1 for t in RELEVANT if t in text_lower)
        irr = sum(1 for t in IRRELEVANT if t in text_lower)
        if rel > 0: score *= min(1.0 + 0.1 * rel, 1.5)
        if irr > 0: score *= max(0.7 ** irr, 0.2)
        reranked.append((score, doc, meta))
    reranked.sort(key=lambda x: x[0], reverse=True)

    context = "\n\n".join([f"[Source: {m['label']}, {m.get('url','')}]\n{d}" for _, d, m in reranked])
    response = (PROMPT | llm).invoke({
        "context": context, "question": question,
        "visa_status": profile.get("visa_status", "F-1"),
        "degree_field": profile.get("degree_field", "Not specified"),
        "graduation_date": profile.get("graduation_date", "Not specified"),
        "employer_type": profile.get("employer_type", "Not specified"),
    })
    return response.content

print("✅ RAG pipeline ready")

✅ RAG pipeline ready


## 6. Launch Gradio App

Builds and launches the conversational UI.

**User profile sidebar:**
- Current visa status (dropdown)
- Degree field — STEM vs non-STEM matters for STEM OPT eligibility
- Graduation date — used by the deadline calculator
- Employer type — affects STEM OPT eligibility

**Example questions** are pre-loaded to guide first-time users.

> `share=True` generates a public URL valid for 72 hours — use this for user testing.

In [6]:
DEADLINE_KEYWORDS = ["deadline", "when", "date", "apply", "timeline", "expire", "window"]

def chat(message, history, visa_status, degree_field, graduation_date, employer_type):
    profile = {"visa_status": visa_status, "degree_field": degree_field,
               "graduation_date": graduation_date, "employer_type": employer_type}
    answer = ask(message, profile)
    if any(kw in message.lower() for kw in DEADLINE_KEYWORDS) and graduation_date:
        try:
            grad_date = datetime.strptime(graduation_date, "%Y-%m-%d").date()
            answer += "\n\n---\n" + format_deadlines(calculate_deadlines(grad_date))
        except ValueError:
            pass
    return answer

with gr.Blocks(title="ImmigrationNavigator") as app:
    gr.Markdown("""
    # 🧭 ImmigrationNavigator
    **AI-powered guidance for the F-1 → OPT → STEM OPT → H-1B pipeline**
    > ⚠️ Guidance based on official USCIS sources. Not legal advice.
    """)
    gr.ChatInterface(
        fn=chat,
        additional_inputs=[
            gr.Dropdown(choices=["F-1 student", "F-1, currently on OPT", "F-1, on STEM OPT", "H-1B pending"],
                        label="Current visa status", value="F-1 student"),
            gr.Dropdown(choices=["Computer Science (STEM)", "Data Science (STEM)", "Engineering (STEM)",
                                 "Biology (STEM)", "Business (non-STEM)", "Humanities (non-STEM)", "Other"],
                        label="Degree field", value="Computer Science (STEM)"),
            gr.Textbox(label="Graduation date (YYYY-MM-DD)", placeholder="e.g. 2025-05-15"),
            gr.Dropdown(choices=["Full-time employer", "Part-time employer", "Multiple employers",
                                 "Consulting firm", "Self-employed", "Not yet employed"],
                        label="Employer type", value="Full-time employer"),
        ],
        examples=[
            ["When do I need to apply for OPT?"],
            ["Am I eligible for STEM OPT extension?"],
            ["What happens during the H-1B cap-gap period?"],
            ["What forms do I need for OPT?"],
            ["How many days can I be unemployed on OPT?"],
        ],
        title="",
    )

app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://3ca377d67ba0e2cce3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
